In [ ]:
!pip install iterative-stratification tqdm scikit-learn numpy torch transformers pandas scikit-multilear datetime

ERROR: Could not find a version that satisfies the requirement scikit-multilear (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: No matching distribution found for scikit-multilear


In [ ]:
import datetime
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support
import numpy as np
from tqdm import tqdm
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_scheduler,
    DataCollatorWithPadding
)
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F

start_time = datetime.datetime.now()
print(f"Script started at {start_time}")

timing_info = {}
timing_info['Start timing'] = start_time

Script started at 2026-04-07 16:36:36.236946


In [ ]:
# Configuration parameters
two_emotions = False
file_name = "GroundTruth.csv"
checkpoint = "bert-base-cased"
num_folds = 5
emotions = ["Joy", "Trust", "Fear", "Surprise", "Sadness", "Disgust", "Anger", "Anticipation", "Neutral"]

# Learning constraints
max_labels = 2

if two_emotions:
    file_name = "GroundTruthJoySadness.csv"
    emotions = ["Joy", "Sadness"]

# Undersampling configuration
undersample = False
cut_off = 150

# Class weight configuration
class_weight = True
function = 0  # options: BCEWithLogitsLoss (0), FocalLoss (1) or CrossEntropyLoss (2)
reduction = "mean" # options: mean or sum
gamma = 2.0  # for Focal Loss, when incrementing the value, focus more on hard examples.

In [ ]:
# Configure output file path
suffixes = []
if undersample:
    suffixes.append("undersample")
if class_weight:
    suffixes.append("class_weight")
    if function == 0:
        suffixes.append("bce_with_logits_loss")
    elif function == 1:
        suffixes.append("focal_loss")
    elif function == 2:
        suffixes.append("cross_entropy_loss")

    suffixes.append(f"reduction_{reduction}")

prefix = "results_2e" if two_emotions else "results_all"

suffix_part = f"_{'_'.join(suffixes)}" if suffixes else ""

# Combine everything
output_path_csv = f"results/{prefix}_max{max_labels}_with_folds_{num_folds}{suffix_part}.csv"

In [ ]:
df = pd.read_csv(file_name, quotechar='"', sep=';')
#df = df[df['Reject'] != 1]  # Filter out rows where Reject is 1

In [ ]:
if undersample:
    start_undersampling = datetime.datetime.now()
    rng = np.random.default_rng(42)
    df = df.reset_index(drop=True)
    pos_counts = df[emotions].sum(axis=0).astype(int)
    targets = np.minimum(cut_off, pos_counts).astype(int)

    selected_idx = set()
    for lbl, tgt in zip(emotions, targets):
        if tgt <= 0:
            continue
        pos_idx = df.index[df[lbl] == 1].to_numpy()
        if len(pos_idx) <= tgt:
            chosen = pos_idx
        else:
            chosen = rng.choice(pos_idx, size=int(tgt), replace=False)
        selected_idx.update(int(x) for x in chosen)

    undersampled = df.loc[sorted(selected_idx)].reset_index(drop=True)
    print("Target per label:", dict(zip(emotions, targets.tolist())))
    print("Result per-label counts:\n", undersampled[emotions].sum().to_dict())
    print("Total rows selected:", len(undersampled))
    df = undersampled
    end_undersampling = datetime.datetime.now()
    timing_info['Undersampling time'] = end_undersampling - start_undersampling

In [ ]:
X = df[['sentence']].values
y = df[emotions].values

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
pos_weight = None

if class_weight:
    start_time_class_weight = datetime.datetime.now()
    num_pos = df[emotions].sum().values
    num_neg = len(df) - num_pos

    weights = num_neg / (num_pos + 1e-6)
    pos_weight = torch.tensor(weights, dtype=torch.float).to(device)

    print("Label counts (Positives):\n", num_pos)
    print("Computed pos_weights (Neg/Pos):\n", pos_weight)

    print("Weights per emotion:")
    for emotion, w in zip(emotions, pos_weight.tolist()):
        print(f"{emotion:12s}: {w:.3f}")

    timing_info['Class weight calculation time'] = datetime.datetime.now() - start_time_class_weight

Label counts (Positives):
 [372 152  24  54 303  50  27 182  88]
Computed pos_weights (Neg/Pos):
 tensor([ 1.9892,  6.3158, 45.3333, 19.5926,  2.6700, 21.2400, 40.1852,  5.1099,
        11.6364])
Weights per emotion:
Joy         : 1.989
Trust       : 6.316
Fear        : 45.333
Surprise    : 19.593
Sadness     : 2.670
Disgust     : 21.240
Anger       : 40.185
Anticipation: 5.110
Neutral     : 11.636


In [ ]:
class MultiLabelFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None, reduction='mean'):
        super(MultiLabelFocalLoss, self).__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(
            inputs,
            targets,
            reduction=reduction,
            pos_weight=self.pos_weight
        )

        pt = torch.exp(-bce_loss)

        focal_loss = ((1 - pt) ** self.gamma) * bce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, label_names):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.label_names = label_names

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        tokenized = self.tokenizer(row["sentence"], truncation=True, padding=False)
        labels = [int(row[l]) for l in self.label_names]
        tokenized["labels"] = labels
        return tokenized

In [ ]:
def collate_fn(batch):
    input_features = [{k: v for k, v in ex.items() if k != "labels"} for ex in batch]
    labels = [ex["labels"] for ex in batch]
    batch_inputs = data_collator(input_features)
    batch_inputs["labels"] = torch.tensor(labels, dtype=torch.float)
    return batch_inputs

In [ ]:
def create_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=len(emotions),
        problem_type="multi_label_classification"
    )
    model.to(device)
    return model

In [ ]:
def get_constrained_predictions(logits):
    """
    Get top-k predictions based on max_labels constraint.

    Args:
        logits: tensor of shape (batch_size, num_labels)

    Returns:
        numpy array of shape (batch_size, num_labels) with 0/1 predictions
    """
    probs = torch.sigmoid(logits)

    # Get top max_labels indices per sample
    num_labels = probs.shape[1]
    k = min(max_labels, num_labels)

    _, top_indices = torch.topk(probs, k=k, dim=1)

    final_preds = torch.zeros_like(probs, dtype=torch.int)

    # Set predicted labels to 1
    for i, indices in enumerate(top_indices):
        final_preds[i, indices] = 1

    return final_preds.numpy()

In [ ]:
multilabel_stratified_in_folds = MultilabelStratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)
all_metrics = []
fold_results = []

In [ ]:
for fold, (train_idx, val_idx) in enumerate(multilabel_stratified_in_folds.split(X, y)):
    timing_fold = datetime.datetime.now()
    print(f"\n===== Fold {fold + 1} =====")
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)

    train_dataset = EmotionDataset(train_df, tokenizer, emotions)
    val_dataset = EmotionDataset(val_df, tokenizer, emotions)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

    model = create_model()

    optimizer = AdamW(model.parameters(), lr=2e-5)
    num_epochs = 3
    num_training_steps = num_epochs * len(train_loader)
    lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0,
                                 num_training_steps=num_training_steps)

    # ---- training loop ----
    model.train()
    progress_bar = tqdm(range(num_training_steps))

    if class_weight and function == 0:
        loss_fct = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction=reduction)
    elif class_weight and function == 1:
        loss_fct = MultiLabelFocalLoss(gamma=2.0, pos_weight=pos_weight, reduction=reduction)
    elif class_weight and function == 2:
        loss_fct = torch.nn.CrossEntropyLoss(weight=pos_weight, reduction=reduction)

    for epoch in range(num_epochs):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            if class_weight:
                logits = outputs.logits
                loss = loss_fct(logits, batch["labels"].float())
            else:
                loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)

    # ---- evaluation ----
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            labels = batch["labels"].cpu().numpy().astype(int)
            inputs = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            outputs = model(**inputs)
            logits = outputs.logits.cpu()
            preds = get_constrained_predictions(logits)
            all_preds.append(preds)
            all_labels.append(labels)

    y_pred = np.vstack(all_preds)
    y_true = np.vstack(all_labels)

    sentences = val_df["sentence"].values


    def binvec_to_names(vec):
        names = [emotions[i] for i, v in enumerate(vec) if int(v) == 1]
        return ", ".join(names) if names else "None"


    true_emotions = [binvec_to_names(row) for row in y_true]
    pred_emotions = [binvec_to_names(row) for row in y_pred]

    df_predictions = pd.DataFrame({
        "sentence": sentences,
        "true_emotions": true_emotions,
        "pred_emotions": pred_emotions
    })

    # show a quick preview and optionally save per fold
    #print(df_predictions.head(20).to_string(index=False))
    df_predictions.to_csv(f"predictions/predictions_fold_{fold + 1}_out_of_{num_folds}.csv", index=False)

    fold_results.append({
        "fold": fold + 1,
        "y_true": y_true,
        "y_pred": y_pred
    })

    metrics = {
        "Subset Accuracy (Exact Match)": accuracy_score(y_true, y_pred),
        "Micro Precision": precision_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro Recall": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "Micro F1": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted Precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "Weighted Recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

    # --- Print the Summary ---
    print(f"\n###### Metrics for fold {fold + 1} ######")
    for k, v in metrics.items():
        print(f"{k:25s}: {v:.4f}")

    all_metrics.append(metrics)
    end_timing_fold = datetime.datetime.now()
    timing_info[f'Fold {fold + 1} time'] = end_timing_fold - timing_fold


===== Fold 1 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 336/336 [04:29<00:00,  2.04it/s]


###### Metrics for fold 1 ######
Subset Accuracy (Exact Match): 0.0045
Micro Precision          : 0.2095
Micro Recall             : 0.3735
Micro F1                 : 0.2684
Macro Precision          : 0.2297
Macro Recall             : 0.4102
Macro F1                 : 0.2365
Weighted Precision       : 0.3802
Weighted Recall          : 0.3735
Weighted F1              : 0.3092

===== Fold 2 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 336/336 [04:35<00:00,  1.22it/s]



###### Metrics for fold 2 ######
Subset Accuracy (Exact Match): 0.0225
Micro Precision          : 0.3671
Micro Recall             : 0.6494
Micro F1                 : 0.4691
Macro Precision          : 0.3260
Macro Recall             : 0.5870
Macro F1                 : 0.3891
Weighted Precision       : 0.4874
Weighted Recall          : 0.6494
Weighted F1              : 0.5296

===== Fold 3 =====


...


###### Metrics for fold 3 ######
Subset Accuracy (Exact Match): 0.0314
Micro Precision          : 0.4013
Micro Recall             : 0.7160
Micro F1                 : 0.5144
Macro Precision          : 0.3196
Macro Recall             : 0.5987
Macro F1                 : 0.4027
Weighted Precision       : 0.4467
Weighted Recall          : 0.7160
Weighted F1              : 0.5391

===== Fold 4 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 336/336 [04:27<00:00,  1.26it/s]



###### Metrics for fold 4 ######
Subset Accuracy (Exact Match): 0.0673
Micro Precision          : 0.4372
Micro Recall             : 0.7708
Micro F1                 : 0.5579
Macro Precision          : 0.3640
Macro Recall             : 0.6059
Macro F1                 : 0.4410
Weighted Precision       : 0.4664
Weighted Recall          : 0.7708
Weighted F1              : 0.5720

===== Fold 5 =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
100%|██████████| 336/336 [04:32<00:00,  1.33it/s]


###### Metrics for fold 5 ######
Subset Accuracy (Exact Match): 0.0270
Micro Precision          : 0.4032
Micro Recall             : 0.7189
Micro F1                 : 0.5166
Macro Precision          : 0.2879
Macro Recall             : 0.4999
Macro F1                 : 0.3566
Weighted Precision       : 0.4475
Weighted Recall          : 0.7189
Weighted F1              : 0.5432


In [ ]:
print("\n###### Summary per fold ######")
timing_results = datetime.datetime.now()

df_metrics = pd.DataFrame(all_metrics).T  # transpose to have metrics as rows
df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
print("\n###### Overall Metrics per Fold ######")
print(df_metrics.to_markdown(floatfmt=".4f"))


###### Summary per fold ######

###### Overall Metrics per Fold ######
|                               |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:------------------------------|---------:|---------:|---------:|---------:|---------:|
| Subset Accuracy (Exact Match) |   0.0045 |   0.0225 |   0.0314 |   0.0673 |   0.0270 |
| Micro Precision               |   0.2095 |   0.3671 |   0.4013 |   0.4372 |   0.4032 |
| Micro Recall                  |   0.3735 |   0.6494 |   0.7160 |   0.7708 |   0.7189 |
| Micro F1                      |   0.2684 |   0.4691 |   0.5144 |   0.5579 |   0.5166 |
| Macro Precision               |   0.2297 |   0.3260 |   0.3196 |   0.3640 |   0.2879 |
| Macro Recall                  |   0.4102 |   0.5870 |   0.5987 |   0.6059 |   0.4999 |
| Macro F1                      |   0.2365 |   0.3891 |   0.4027 |   0.4410 |   0.3566 |
| Weighted Precision            |   0.3802 |   0.4874 |   0.4467 |   0.4664 |   0.4475 |
| Weighted Recall               |   0.

In [ ]:
emotion_metrics = {emotion: {"Precision": [], "Recall": [], "F1": [], "Accuracy": []} for emotion in emotions}

for fold_data in fold_results:
    y_true = fold_data["y_true"]
    y_pred = fold_data["y_pred"]
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0
    )

    acc = [accuracy_score(y_true[:, i], y_pred[:, i]) for i in range(y_true.shape[1])]

    for i, emotion in enumerate(emotions):
        emotion_metrics[emotion]["Precision"].append(prec[i])
        emotion_metrics[emotion]["Recall"].append(rec[i])
        emotion_metrics[emotion]["F1"].append(f1[i])
        emotion_metrics[emotion]["Accuracy"].append(acc[i])

In [ ]:
print("\n###### Per Emotion Metrics per Fold ######")
for emotion, metrics_dict in emotion_metrics.items():
    df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
    print(f"\n--- {emotion} ---")
    print(df_emotion.to_markdown(floatfmt=".4f"))


###### Per Emotion Metrics per Fold ######

--- Joy ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|---------:|---------:|---------:|---------:|---------:|
| Precision |   0.7586 |   0.6620 |   0.4878 |   0.5349 |   0.5317 |
| Recall    |   0.2973 |   0.6351 |   0.8108 |   0.9200 |   0.8933 |
| F1        |   0.4272 |   0.6483 |   0.6091 |   0.6765 |   0.6667 |
| Accuracy  |   0.7342 |   0.7703 |   0.6547 |   0.7040 |   0.6982 |

--- Trust ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|---------:|---------:|---------:|---------:|---------:|
| Precision |   0.2150 |   0.2177 |   0.3030 |   0.3295 |   0.3165 |
| Recall    |   0.7419 |   0.8710 |   0.6667 |   0.9667 |   0.8333 |
| F1        |   0.3333 |   0.3484 |   0.4167 |   0.4915 |   0.4587 |
| Accuracy  |   0.5856 |   0.5450 |   0.7489 |   0.7309 |   0.7342 |

--- Fear ---
|           |   Fold 1 |   Fold 2 |   Fold 3 |   Fold 4 |   Fold 5 |
|:----------|----

In [ ]:
print("\n###### Average Per-Emotion Metrics ######")
df_emotion_avg = pd.DataFrame({
    emotion: {metric: np.mean(values) for metric, values in metrics.items()}
    for emotion, metrics in emotion_metrics.items()
})
print(df_emotion_avg.T.to_markdown(floatfmt=".4f"))


###### Average Per-Emotion Metrics ######
|              |   Precision |   Recall |     F1 |   Accuracy |
|:-------------|------------:|---------:|-------:|-----------:|
| Joy          |      0.5950 |   0.7113 | 0.6055 |     0.7123 |
| Trust        |      0.2763 |   0.8159 | 0.4097 |     0.6689 |
| Fear         |      0.2088 |   0.5000 | 0.2622 |     0.9325 |
| Surprise     |      0.1759 |   0.3873 | 0.2268 |     0.8389 |
| Sadness      |      0.5517 |   0.6495 | 0.5953 |     0.7643 |
| Disgust      |      0.1468 |   0.4200 | 0.2019 |     0.8633 |
| Anger        |      0.1406 |   0.2933 | 0.1870 |     0.9380 |
| Anticipation |      0.4696 |   0.6964 | 0.5559 |     0.8210 |
| Neutral      |      0.1843 |   0.3895 | 0.2424 |     0.7894 |


In [ ]:
df_metrics = pd.DataFrame(all_metrics)
print("\n###### Cross-validation mean results ######")
print(df_metrics.mean())

timing_info[f'Printing results'] = datetime.datetime.now() - timing_results

with open(output_path_csv, mode="w", newline="") as f:
    writer = csv.writer(f, delimiter=";")

    # ===== SUMMARY PER FOLD ######
    writer.writerow(["###### Summary per fold ######"])
    df_metrics = pd.DataFrame(all_metrics).T  # metrics as rows
    df_metrics.columns = [f"Fold {i + 1}" for i in range(len(all_metrics))]
    df_metrics.reset_index(inplace=True)
    df_metrics.rename(columns={"index": "Metric"}, inplace=True)
    df_metrics.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")
    writer.writerow([])  # blank line

    # ###### PER-EMOTION METRICS PER FOLD ######
    writer.writerow(["###### Per Emotion Metrics per Fold ######"])
    for emotion, metrics_dict in emotion_metrics.items():
        writer.writerow([f"--- {emotion} ---"])
        df_emotion = pd.DataFrame(metrics_dict, index=[f"Fold {i + 1}" for i in range(len(fold_results))]).T
        df_emotion.reset_index(inplace=True)
        df_emotion.rename(columns={"index": "Metric"}, inplace=True)
        df_emotion.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")
        writer.writerow([])
        writer.writerow([])  # blank line

    # ###### AVERAGE PER EMOTION ######
    writer.writerow(["###### Average Per-Emotion Metrics ######"])
    df_emotion_avg = pd.DataFrame({
        emotion: {metric: np.mean(values) for metric, values in metrics.items()}
        for emotion, metrics in emotion_metrics.items()
    })
    df_emotion_avg = df_emotion_avg.T.reset_index().rename(columns={"index": "Emotion"})
    df_emotion_avg.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")
    writer.writerow([])

    # ###### CROSS-VALIDATION MEAN RESULTS ######
    writer.writerow(["###### Cross-validation Mean Results ######"])
    df_metrics = pd.DataFrame(all_metrics)
    df_mean = df_metrics.mean().to_frame(name="Mean").reset_index().rename(columns={"index": "Metric"})
    df_mean.to_csv(f, index=False, float_format="%.4f", sep=";", decimal=",")

    # ###### CLASS WEIGHTS ######
    if class_weight:
        writer.writerow([])
        writer.writerow(["###### Class Weights per Emotion ######"])
        writer.writerow(["Emotion", "Weight"])
        for emotion, weight in zip(emotions, pos_weight.tolist()):
            writer.writerow([emotion, f"{weight:.4f}"])
        writer.writerow([])

    writer.writerow(["###### Timings ######"])
    for k, v in timing_info.items():
        writer.writerow([k, str(v)])

    writer.writerow(["Total duration", str(datetime.datetime.now() - start_time)])

print(f"\n✅ All tables saved in one CSV file: {os.path.abspath(output_path_csv)}")


###### Cross-validation mean results ######
Subset Accuracy (Exact Match)    0.030542
Micro Precision                  0.363659
Micro Recall                     0.645705
Micro F1                         0.465273
Macro Precision                  0.305448
Macro Recall                     0.540359
Macro F1                         0.365194
Weighted Precision               0.445650
Weighted Recall                  0.645705
Weighted F1                      0.498625
dtype: float64

✅ All tables saved in one CSV file: /Users/carlotacatot/Docs/pHd/EmotionClassification/results/results_all_max2_with_folds_5_class_weight_bce_with_logits_loss_reduction_mean.csv


In [ ]:
print('\n===== Training Final Model on Full Dataset =====')
import datetime

timing_final_start = datetime.datetime.now()

# Create full dataset and data loader
full_dataset = EmotionDataset(df, tokenizer, emotions)
full_loader = DataLoader(full_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

# Instantiate a fresh model
final_model = create_model()

num_epochs_final = 3
num_training_steps_final = num_epochs_final * len(full_loader)

# Define optimizer and scheduler
optimizer_final = AdamW(final_model.parameters(), lr=2e-5)
lr_scheduler_final = get_scheduler(
    'linear',
    optimizer=optimizer_final,
    num_warmup_steps=0,
    num_training_steps=num_training_steps_final
)

# Determine the loss function
if class_weight and function == 0:
    loss_fct_final = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction=reduction)
elif class_weight and function == 1:
    loss_fct_final = MultiLabelFocalLoss(gamma=gamma, pos_weight=pos_weight, reduction=reduction)
elif class_weight and function == 2:
    loss_fct_final = torch.nn.CrossEntropyLoss(weight=pos_weight, reduction=reduction)
else:
    loss_fct_final = None

# Training Loop
final_model.train()
progress_bar_final = tqdm(range(num_training_steps_final), desc='Final training')

for epoch in range(num_epochs_final):
    for batch in full_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = final_model(**batch)
        
        if loss_fct_final is not None:
            logits = outputs.logits
            loss = loss_fct_final(logits, batch['labels'].float())
        else:
            loss = outputs.loss
            
        loss.backward()
        optimizer_final.step()
        lr_scheduler_final.step()
        optimizer_final.zero_grad()
        progress_bar_final.update(1)

timing_info['Final model training time'] = datetime.datetime.now() - timing_final_start
print(f"\n✅ Final model trained in {timing_info['Final model training time']}")


===== Training Final Model on Full Dataset =====


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



✅ Final model trained in 0:06:21.633500


In [ ]:
import os
import json

# Build a descriptive directory name mirroring the CSV output prefix
model_dir = f"../models/dataimbalanced_{suffix_part}"
os.makedirs(model_dir, exist_ok=True)

# Save model weights and configuration
final_model.save_pretrained(model_dir)

# Save tokenizer so the model is fully self-contained
tokenizer.save_pretrained(model_dir)

# Save a small metadata file for reproducibility
meta = {
    'base_checkpoint': checkpoint,
    'emotions': emotions,
    'num_folds': num_folds,
    'num_epochs': num_epochs_final,
    'learning_rate': 2e-5,
    'batch_size': 8,
    'max_labels': max_labels,
    'undersample': undersample,
    'cut_off': cut_off if undersample else None,
    'class_weight': class_weight,
    'reduction': reduction,
    'trained_on': 'full_dataset',
    'training_samples': len(df),
    'saved_at': str(datetime.datetime.now()),
}

with open(os.path.join(model_dir, 'training_meta.json'), 'w') as _f:
    json.dump(meta, _f, indent=2)

print(f"\n✅ Final model saved to: {os.path.abspath(model_dir)}")
print("\nTo reload the model later in another script, use:")
print(f"   tokenizer = AutoTokenizer.from_pretrained(\"{model_dir}\")")
print(f"   model = AutoModelForSequenceClassification.from_pretrained(\"{model_dir}\")")


✅ Final model saved to: /Users/carlotacatot/Docs/pHd/EmotionClassification/models/results_all_max2_folds5_class_weight_bce_with_logits_loss_reduction_mean

To reload the model later in another script, use:
   tokenizer = AutoTokenizer.from_pretrained("models/results_all_max2_folds5_class_weight_bce_with_logits_loss_reduction_mean")
   model = AutoModelForSequenceClassification.from_pretrained("models/results_all_max2_folds5_class_weight_bce_with_logits_loss_reduction_mean")
